# Phase 7 -- Historical Replay and Validation

Alert Intelligence Engine -- master plan Phase 7 (section 20): "Time-forward/group evaluation + report." Gate: **PoC evidence approved** -- the last gate before API work begins.

This phase does not train new champions -- it stress-tests and documents the Phase 5 (Entity) and Phase 6 (Transaction) champions:
1. **Reproducibility** -- same model + same input must produce identical scores (master plan: "Model scores are reproducible for a fixed model and feature version").
2. **Validation manifests** -- exact record-level train/test membership persisted (master plan section 10 requirement, not done explicitly until now).
3. **Chronological walk-forward replay** -- a more thorough version of Phase 5/6's single time-forward split: retrain on everything-before, score the next period, repeat across the whole history. Simulates "if this had been deployed and periodically retrained."
4. **Definition of Done checklist** (master plan section 22) -- go through every PoC acceptance item with a reference to actual evidence, not an assertion.

**PII discipline unchanged**: no raw name/DOB/UIN/Customer Number/Beneficiary Name in any output here either.

In [1]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO_ROOT))

import json
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 160)

## 1. Rebuild datasets (identical to Phase 5 / Phase 6)

In [2]:
from pipelines.normalization.pipeline import run_phase2_pipeline
from pipelines.entity.combined_dataset import build_combined_entity_dataset

normalized_sheets, phase2_report = run_phase2_pipeline(REPO_ROOT, persist=False)
assert phase2_report["overall_status"] == "PASS"

combined_entity = build_combined_entity_dataset(
    normalized_sheets["CustomerViolation"], normalized_sheets["TransactionNameViolation"]
)
rule_df = normalized_sheets["Rule"]
print(f"Entity (combined): {len(combined_entity)} rows")
print(f"Transaction (Rule): {len(rule_df)} rows")

phase5_champion = json.load(open(REPO_ROOT / "evaluation" / "phase5_entity_experiments_report.json"))["champion"]
phase6_champion = json.load(open(REPO_ROOT / "evaluation" / "phase6_transaction_experiments_report.json"))["champion"]
print("\nPhase 5 champion:", phase5_champion)
print("Phase 6 champion:", phase6_champion)

Entity (combined): 4397 rows
Transaction (Rule): 2244 rows

Phase 5 champion: {'experiment_id': 'unseen_customers::E5_ocsvm_name_svd', 'model_kind': 'ocsvm', 'representation': 'name_svd', 'validation_scenario': 'unseen_customers'}
Phase 6 champion: {'experiment_id': 'unseen_customers::T5_ocsvm_behavioural_svd', 'model_kind': 'ocsvm', 'representation': 'behavioural_svd', 'validation_scenario': 'unseen_customers'}


## 2. Reproducibility check

Fixed model + fixed input must produce identical scores. Refits each champion's exact recipe twice on the same unseen-customers split and compares.

In [3]:
from pipelines.entity.validation_splits import group_split_by_customer
from features.entity_features import fit_entity_feature_artifacts, transform_entity_features
from pipelines.entity.anomaly_models import extract_name_representation_matrix, fit_svd, fit_ocsvm, score_ocsvm

def entity_champion_scores(seed_note=""):
    train_idx, test_idx = group_split_by_customer(combined_entity, test_size=0.25, random_state=42)
    train_df = combined_entity.iloc[train_idx].reset_index(drop=True)
    test_df = combined_entity.iloc[test_idx].reset_index(drop=True)

    artifacts = fit_entity_feature_artifacts(train_df, "CombinedEntity")
    train_matrix, block_names = transform_entity_features(train_df, "CombinedEntity", artifacts)
    artifacts.feature_names = block_names
    test_matrix, _ = transform_entity_features(test_df, "CombinedEntity", artifacts)

    name_train = extract_name_representation_matrix(train_matrix, artifacts)
    name_test = extract_name_representation_matrix(test_matrix, artifacts)
    svd = fit_svd(name_train, n_components=50)
    model = fit_ocsvm(svd.transform(name_train))
    return score_ocsvm(model, svd.transform(name_test)), train_idx, test_idx

scores_run1, train_idx, test_idx = entity_champion_scores()
scores_run2, _, _ = entity_champion_scores()

max_abs_diff = np.max(np.abs(scores_run1 - scores_run2))
print(f"Entity champion (OCSVM): max |score_run1 - score_run2| = {max_abs_diff:.2e}")
assert max_abs_diff < 1e-9, "Champion scores are not reproducible for fixed model + fixed input"
print("PASS: bit-for-bit reproducible.")

Entity champion (OCSVM): max |score_run1 - score_run2| = 0.00e+00
PASS: bit-for-bit reproducible.


In [4]:
from features.transaction_features import fit_transaction_feature_artifacts, transform_transaction_features
from pipelines.transaction.anomaly_models import extract_structured_matrix  # noqa: F401 (T1 not needed here, champion is T5)

def transaction_champion_scores():
    train_idx_t, test_idx_t = group_split_by_customer(rule_df, test_size=0.25, random_state=42)
    train_df = rule_df.iloc[train_idx_t].reset_index(drop=True)
    test_df = rule_df.iloc[test_idx_t].reset_index(drop=True)

    artifacts = fit_transaction_feature_artifacts(train_df)
    train_matrix, block_names = transform_transaction_features(train_df, artifacts)
    artifacts.feature_names = block_names
    test_matrix, _ = transform_transaction_features(test_df, artifacts)

    svd = fit_svd(train_matrix, n_components=50)
    model = fit_ocsvm(svd.transform(train_matrix))
    return score_ocsvm(model, svd.transform(test_matrix)), train_idx_t, test_idx_t

t_scores_run1, t_train_idx, t_test_idx = transaction_champion_scores()
t_scores_run2, _, _ = transaction_champion_scores()

max_abs_diff_t = np.max(np.abs(t_scores_run1 - t_scores_run2))
print(f"Transaction champion (OCSVM): max |score_run1 - score_run2| = {max_abs_diff_t:.2e}")
assert max_abs_diff_t < 1e-9, "Champion scores are not reproducible for fixed model + fixed input"
print("PASS: bit-for-bit reproducible.")

Transaction champion (OCSVM): max |score_run1 - score_run2| = 0.00e+00
PASS: bit-for-bit reproducible.


## 3. Validation manifests

Exact record-level train/test membership, persisted for audit (master plan section 10). `record_id`/`customer_id` are opaque hashes/namespaced strings -- safe to commit, not PII.

In [5]:
from pipelines.evaluation.validation_manifest import (
    build_validation_manifest, save_validation_manifest, verify_no_customer_leakage,
)

manifest_dir = REPO_ROOT / "evaluation" / "validation_manifests"

entity_manifest = build_validation_manifest(
    combined_entity, train_idx, test_idx,
    scenario_name="phase5_entity_unseen_customers",
    dataset_version=phase2_report["dataset_version"],
    feature_version="entity-v1",
)
entity_manifest_path = save_validation_manifest(entity_manifest, manifest_dir)
print(f"Entity manifest: {entity_manifest_path}")
print(f"  n_train={len(entity_manifest.train_record_ids)}, n_test={len(entity_manifest.test_record_ids)}")
print(f"  no customer leakage: {verify_no_customer_leakage(entity_manifest)} (expected True -- group split)")

transaction_manifest = build_validation_manifest(
    rule_df, t_train_idx, t_test_idx,
    scenario_name="phase6_transaction_unseen_customers",
    dataset_version=phase2_report["dataset_version"],
    feature_version="transaction-v1",
)
transaction_manifest_path = save_validation_manifest(transaction_manifest, manifest_dir)
print(f"\nTransaction manifest: {transaction_manifest_path}")
print(f"  n_train={len(transaction_manifest.train_record_ids)}, n_test={len(transaction_manifest.test_record_ids)}")
print(f"  no customer leakage: {verify_no_customer_leakage(transaction_manifest)} (expected True -- group split)")

Entity manifest: /home/chpl/Documents/AI-validation/AI_validator/Alert-AI/evaluation/validation_manifests/phase5_entity_unseen_customers_manifest.json
  n_train=3376, n_test=1021
  no customer leakage: True (expected True -- group split)

Transaction manifest: /home/chpl/Documents/AI-validation/AI_validator/Alert-AI/evaluation/validation_manifests/phase6_transaction_unseen_customers_manifest.json
  n_train=1599, n_test=596
  no customer leakage: True (expected True -- group split)


## 4. Chronological walk-forward replay

More thorough than Phase 5/6's single time-forward split: retrain on everything strictly before each period, score that period, repeat across the full history. This simulates periodic retraining in production.

In [6]:
from pipelines.evaluation.walk_forward import run_walk_forward

def entity_fit_transform_fn(train_df, score_df):
    artifacts = fit_entity_feature_artifacts(train_df, "CombinedEntity")
    train_matrix, block_names = transform_entity_features(train_df, "CombinedEntity", artifacts)
    artifacts.feature_names = block_names
    score_matrix, _ = transform_entity_features(score_df, "CombinedEntity", artifacts)
    name_train = extract_name_representation_matrix(train_matrix, artifacts)
    name_score = extract_name_representation_matrix(score_matrix, artifacts)
    svd = fit_svd(name_train, n_components=50)
    return svd.transform(name_train), svd.transform(name_score)

entity_walk_forward = run_walk_forward(
    combined_entity, "Alert Generated Date & Time (Parsed)",
    entity_fit_transform_fn, fit_ocsvm, score_ocsvm, n_folds=4, min_train_size=50,
)
print("=== Entity champion (OCSVM) walk-forward ===")
for r in entity_walk_forward:
    print(r)

=== Entity champion (OCSVM) walk-forward ===
{'fold_index': 0, 'skipped': True, 'reason': 'train size 0 < min_train_size 50', 'n_train': 0, 'n_score': 1100}
{'fold_index': 1, 'skipped': False, 'n_train': 1100, 'n_score': 1099, 'score_mean': -17.715272947127907, 'score_std': 4.319832924303881, 'score_min': -27.01943553725651, 'score_max': -9.339527713680532}
{'fold_index': 2, 'skipped': False, 'n_train': 2199, 'n_score': 1099, 'score_mean': -35.243641234165054, 'score_std': 8.298274601763628, 'score_min': -48.51640487386404, 'score_max': -14.14437524540434}
{'fold_index': 3, 'skipped': False, 'n_train': 3298, 'n_score': 1099, 'score_mean': -41.59220550271124, 'score_std': 10.481880870111874, 'score_min': -58.374988628701786, 'score_max': -16.669246710010952}


In [7]:
def transaction_fit_transform_fn(train_df, score_df):
    artifacts = fit_transaction_feature_artifacts(train_df)
    train_matrix, block_names = transform_transaction_features(train_df, artifacts)
    artifacts.feature_names = block_names
    score_matrix, _ = transform_transaction_features(score_df, artifacts)
    svd = fit_svd(train_matrix, n_components=50)
    return svd.transform(train_matrix), svd.transform(score_matrix)

transaction_walk_forward = run_walk_forward(
    rule_df, "Scan Date (Parsed)",
    transaction_fit_transform_fn, fit_ocsvm, score_ocsvm, n_folds=4, min_train_size=50,
)
print("=== Transaction champion (OCSVM) walk-forward ===")
for r in transaction_walk_forward:
    print(r)

=== Transaction champion (OCSVM) walk-forward ===
{'fold_index': 0, 'skipped': True, 'reason': 'train size 0 < min_train_size 50', 'n_train': 0, 'n_score': 561}
{'fold_index': 1, 'skipped': False, 'n_train': 561, 'n_score': 561, 'score_mean': -12.841213405913418, 'score_std': 1.0643941876688845, 'score_min': -15.158761898799115, 'score_max': -2.0936759014241826}
{'fold_index': 2, 'skipped': False, 'n_train': 1122, 'n_score': 561, 'score_mean': -24.80377787319899, 'score_std': 7.013276747856215, 'score_min': -30.571039098045723, 'score_max': -0.0}
{'fold_index': 3, 'skipped': False, 'n_train': 1683, 'n_score': 561, 'score_mean': -47.93313190175505, 'score_std': 0.8165746381082845, 'score_min': -50.92261412303953, 'score_max': -45.70033382720367}


### Finding: OCSVM's raw score scale is not stationary across retrains

Both walk-forwards show `score_mean`/`score_std` drifting as the training population grows fold-over-fold (not converging to a stable range). This is expected behaviour for `OneClassSVM.score_samples` -- its scale depends on the specific fitted hyperplane, which shifts as training data changes -- **not a sign the model is broken**, but it is decisive evidence for a specific production control:

**Raw anomaly scores must never be used as a fixed operational threshold.** Master plan Appendix B: "Never convert anomaly score directly into a compliance probability without calibration data," and section 13: "Any operational routing threshold must be calibrated and approved; not hardcoded from a sample guess." A threshold tuned against fold 1's score range would misfire against fold 3's. Any production routing threshold needs a calibration step (e.g. percentile-of-current-population rather than an absolute score cutoff) -- out of scope for this PoC, flagged for Phase 8+ (API) design.

## 5. Definition of Done -- master plan section 22

In [8]:
dod_checklist = [
    ("The three alert types are supported.", "PASS", "Phase 1-4: schema, normalization, features for CustomerViolation, TransactionNameViolation, Rule"),
    ("The current XLSX can be processed end-to-end.", "PASS", "Phase 1-2: load_raw_alerts -> run_phase2_pipeline, 6,641 rows, 0 schema failures"),
    ("No LLM or paid AI API is required.", "PASS", "scikit-learn/PyTorch only, verified by dependency list (requirements.txt)"),
    ("No hardcoded matching-weight decision engine exists.", "PASS", "No RapidFuzz/Levenshtein/manual weights anywhere in pipelines/ or features/"),
    ("No post-review leakage fields enter live inference.", "PASS", "Phase 1 leakage registry + Phase 3/4 static import-time guards + tests"),
    ("Duplicate and repeated-entity leakage is controlled.", "PASS", "Phase 2: exact dup detection + near-dup candidates; group-split validation"),
    ("Entity and transaction pipelines are separated.", "PASS", "features/entity_features.py vs features/transaction_features.py, separate champions"),
    ("At least Isolation Forest and Autoencoder have been fairly benchmarked.", "PASS", "Phase 5 (E1-E5) and Phase 6 (T1-T5), both under 2 validation scenarios"),
    ("A champion model is selected using documented evidence.", "PASS", "pipelines/entity/evaluation.py:select_champion, rubric persisted in both reports"),
    ("Historical replay is reproducible.", "PASS", "Section 2 above -- bit-for-bit identical scores confirmed for both champions"),
    ("Top-N novelty ranking can be demonstrated.", "PASS", "Phase 5/6 section 7/8 top_n_novelty_table"),
    ("Customer-specific historical context is demonstrated where data permits.", "PASS", "Phase 3/4 build_historical_customer_context, Phase 5/6 customer_history_consistency"),
    ("Rule-specific analysis is demonstrated.", "PASS", "Phase 6 section 7 novelty_by_segment(..., 'Rule Name')"),
    ("The scoring API works for single and batch alerts.", "NOT YET", "Phase 9 (API implementation) -- not started"),
    ("Every score is auditable and versioned.", "PARTIAL", "Feature/model versions exist on artifacts; API-level score audit record is Phase 9"),
    ("The dashboard demonstrates the client's future workflow.", "NOT YET", "Phase 10 (Demo UI) -- not started"),
    ("Feedback can be recorded.", "NOT YET", "Phase 11 (feedback/monitoring foundations) -- not started"),
    ("Monitoring and retraining foundations exist.", "NOT YET", "Phase 11 -- not started"),
    ("The report clearly separates exploratory anomaly findings from confirmed supervised performance.", "PASS", "Every phase report + this notebook's section 4 finding; exploratory_status_comparison always labeled"),
]

print(f"{'STATUS':<10} ITEM")
for item, status, evidence in dod_checklist:
    print(f"{status:<10} {item}")
    print(f"           evidence: {evidence}")
print()
n_pass = sum(1 for _, s, _ in dod_checklist if s == "PASS")
print(f"{n_pass}/{len(dod_checklist)} items PASS -- remainder are Phase 8+ (API/dashboard/feedback), not PoC-stage gaps.")

STATUS     ITEM
PASS       The three alert types are supported.
           evidence: Phase 1-4: schema, normalization, features for CustomerViolation, TransactionNameViolation, Rule
PASS       The current XLSX can be processed end-to-end.
           evidence: Phase 1-2: load_raw_alerts -> run_phase2_pipeline, 6,641 rows, 0 schema failures
PASS       No LLM or paid AI API is required.
           evidence: scikit-learn/PyTorch only, verified by dependency list (requirements.txt)
PASS       No hardcoded matching-weight decision engine exists.
           evidence: No RapidFuzz/Levenshtein/manual weights anywhere in pipelines/ or features/
PASS       No post-review leakage fields enter live inference.
           evidence: Phase 1 leakage registry + Phase 3/4 static import-time guards + tests
PASS       Duplicate and repeated-entity leakage is controlled.
           evidence: Phase 2: exact dup detection + near-dup candidates; group-split validation
PASS       Entity and transaction pipeline

## 6. Phase 7 report

In [9]:
phase7_report = {
    "phase": "7_historical_replay",
    "status": "PASS",
    "reproducibility": {
        "entity_champion_max_abs_score_diff": float(max_abs_diff),
        "transaction_champion_max_abs_score_diff": float(max_abs_diff_t),
        "both_bit_for_bit_reproducible": True,
    },
    "validation_manifests": {
        "entity": str(entity_manifest_path.relative_to(REPO_ROOT)),
        "transaction": str(transaction_manifest_path.relative_to(REPO_ROOT)),
    },
    "walk_forward": {
        "entity": entity_walk_forward,
        "transaction": transaction_walk_forward,
    },
    "findings": [
        "OCSVM raw score scale drifts across walk-forward retrains for both champions -- "
        "not a defect, but decisive evidence that any production routing threshold must be "
        "calibrated against the current population, never a hardcoded absolute score cutoff.",
    ],
    "definition_of_done": [
        {"item": item, "status": status, "evidence": evidence}
        for item, status, evidence in dod_checklist
    ],
    "next_gate": "Phase 8 -- FastAPI scoring implementation (single + batch scoring, model info "
        "endpoints). Requires human approval.",
}

out_path = REPO_ROOT / "evaluation" / "phase7_historical_replay_report.json"
with open(out_path, "w") as f:
    json.dump(phase7_report, f, indent=2, default=str)
print(f"Report written to {out_path}")

Report written to /home/chpl/Documents/AI-validation/AI_validator/Alert-AI/evaluation/phase7_historical_replay_report.json


## 7. Test suite

In [10]:
import subprocess

result = subprocess.run(["python", "-m", "pytest", "tests/", "-q"], cwd=REPO_ROOT, capture_output=True, text=True)
print(result.stdout[-2000:])
if result.returncode != 0:
    print(result.stderr[-2000:])
assert result.returncode == 0, "Test suite must pass before Phase 7 is considered done" 

........................................................................ [ 55%]
.........................................................                [100%]
=============================== warnings summary ===============================
tests/test_anomaly_models.py::test_autoencoder_fit_score_shapes
  /home/chpl/Documents/AI-validation/AI_validator/Alert-AI/.venv/lib/python3.12/site-packages/torch/cuda/__init__.py:619: UserWarning: Can't initialize NVML
    warnings.warn("Can't initialize NVML")

-- Docs: https://docs.pytest.org/en/stable/how-to/capture-warnings.html
129 passed, 1 warning in 12.02s



## Phase 7 -- Result

**Status: PASS**

- Both champions (Entity: OCSVM/name-SVD, Transaction: OCSVM/behavioural-SVD) confirmed bit-for-bit reproducible for fixed model + fixed input.
- Validation manifests persisted with exact record-level train/test membership (master plan section 10 requirement, satisfied explicitly for the first time this phase).
- Chronological walk-forward replay run for both champions -- surfaced a real, non-obvious finding: OCSVM's raw score scale is not stationary across retrains, which is decisive evidence for the threshold-calibration production control already required by the master plan, not a new problem invented here.
- Definition of Done checklist: 13/19 items PASS at PoC stage; the remaining 6 are explicitly Phase 8+ (API, dashboard, feedback/monitoring) and were never in scope for Phases 1-7.
- No PII in any notebook output.

**Next gate:** Phase 8 -- FastAPI scoring implementation (`POST /api/v1/alerts/score`, batch scoring, model-info endpoints) per master plan section 13's API contract. This is the PoC-evidence-approved boundary the master plan describes -- everything before this point was audit/pipeline/model work; Phase 8 starts the client-facing service. Awaiting human approval to proceed.